In [1]:
import pandas as pd
df =pd.read_csv("IMDB Dataset.csv")


In [2]:
df.drop_duplicates(inplace=True)

# pre-processing

In [4]:
#convertinng to lower case
df["review"]=df["review"].str.lower()

In [5]:
### removing the url
import re

def remove_url(text):
    text=re.sub(r"http\s+","",text)
    return text
df["review"]=df["review"].apply(remove_url)

In [6]:
#remove punctuations
def remove_punch(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"]=df["review"].apply(remove_punch)

In [7]:
#remove html tags
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(remove_html)

In [8]:
###removing the stop wards
import nltk #natural language toolkit

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nomaa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nomaa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nomaa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [25]:
def remove_words(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")


    for word in tokens:
        if word in stop_words:
            text=text.replace(word, "")

    return text

df["review"]=df["review"].apply(remove_words)

In [32]:
# stemming
#ex
#running->run
#coding->code 

from nltk.stem import PorterStemmer

In [34]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [38]:
#Encodding

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [57]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

x = tf.fit_transform(df["review"])

y=df["sentiment"] 


,review,sentiment
0,e revew nte wtchg 1 oz epo hook rght exctli hp...,1
1,wder ltle prducti br br film techniqu unssum l...,1
2,hugh h wer w pen me h ummer weeken ng n r cne ...,1
3,bci fli e boy jke hk zob cloe pn fghg ebr br o...,0
4,peer me love i vui unng film wch mr mei fer u ...,1
...,...,...
49995,hough h ove dd rgh good job wn crev gnl fr w e...,1
49996,bd plo bd dlogu bd cng doc drecng nnoyng porn ...,0
49997,cholc ugh n prochl elenri chool nun ugh jeu p ...,0
49998,im gog dge pviou comnt ide mlt e ecd rte exc v...,0


# dataset and loaders

In [59]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 3365201 stored elements in Compressed Sparse Row format>

In [65]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [61]:
x_train=x_train.toarray()
x_test=x_test.toarray()

In [68]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [70]:
train_loader=DataLoader(train_set, shuffle=True, batch_size=64)
test_loader=DataLoader(test_set, shuffle=True, batch_size=64)

# Build Our RNN

In [85]:
import torch.nn as nn
import torch.optim as  optim

In [75]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.rnn=nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True  
        )

# fully connect layer 
        self.fc=nn.Linear(hidden_size, 1)
    def forward(self,x):
        h0=torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out,_=self.rnn(x,h0)
        #1st val = hidden state os all timesteps
        #2nd val = final hidden size of last timestep 


        out=self.fc(out[:,-1,:])
        return out

In [89]:
input_size=x_train.shape[1]
model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

# training the model

In [96]:
epochs=10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"{epoch}/{epochs} and loss = {loss.item()}")

0/10 and loss = 0.30039772391319275
1/10 and loss = 0.4390866458415985
2/10 and loss = 0.29120832681655884
3/10 and loss = 0.35948652029037476
4/10 and loss = 0.37248215079307556
5/10 and loss = 0.3090864419937134
6/10 and loss = 0.2154686003923416
7/10 and loss = 0.639634907245636
8/10 and loss = 0.31794169545173645
9/10 and loss = 0.32957446575164795


In [106]:
model.eval()

with torch.no_grad():
    correct_vals=0
    tot_vals=0
    for xb, yb in test_loader:
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)
        correct_vals+=(predicted == yb).sum().item()
    print(f"accuracy={correct_vals/tot_vals*100}")

accuracy=82.74679842694364
